# 2. Improved BM25 with stemming

Strong accepted lexical reference; low correct negative abstention.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(2)
model['config_snapshot']

{'name': 'bm25_focused_stemmed',
 'chunk_size': 1000,
 'chunk_overlap': 200,
 'k1': 1.5,
 'b': 0.75,
 'min_score': 0.0,
 'k_values': [1, 3, 5, 10],
 'query_mode': 'document_then_clause',
 'clause_query_cleanup': True,
 'clause_cleanup_fallback': True,
 'stemming': 'english'}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.170036,0.415261,0.507143
1,3,140,0.106619,0.606944,0.685714
2,5,140,0.078798,0.670075,0.742857
3,10,140,0.055810,0.753536,0.828571


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 0}
False answerable abstention: 0.0
Cached latency: {'median': 7.58655002573505, 'p95': 10.498929925961418}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
2 passages retrieved.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


Full code walkthrough: ../experiments/09_combined_bm25_stemming.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.